In [2]:
import json
import os
import sys
from pathlib import Path

os.environ["MESA_DIR"] = "/Users/mchau/mesa"   # Replace if your MESA is elsewhere

print("MESA_DIR =", os.environ["MESA_DIR"])

def locate_project_root(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'tutorial' / 'inlist_project').is_file():
            return candidate
    raise FileNotFoundError('Could not locate tutorial/inlist_project')

PROJECT_ROOT = locate_project_root()
PIPELINE_DIR = PROJECT_ROOT / 'scripts' / 'mams_profiles' / 'pipeline'
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from mams_pipeline import (
    build_configs, inspect_runs, model_label, prepare_grid,
    render_inlist_project, run_local, submit_slurm,
)

print(f'Project root: {PROJECT_ROOT}')

# Main user-editable parameters
MASSES_MSUN = [8]
INITIAL_Y = 0.2703
INITIAL_Z = 0.0142
STAGE = "RGB"
CENTRAL_H1_LIMIT = 0.01
RGB_RADIUS_LIMIT = 1.9031


# Slurm settings used only if SUBMIT_SLURM is enabled.
PARTITION = 'typhon'
WALLTIME = '7-00:00:00'
CPUS_PER_TASK = 1

# An environment override makes validation and alternate output locations easy.
OUTPUT_ROOT = Path(os.environ.get(
    'MAMS_OUTPUT_ROOT', PROJECT_ROOT / 'results' / 'mams_profiles'
)).resolve()
TEMPLATE_DIR = PROJECT_ROOT / 'tutorial'
OVERWRITE = False
PREPARE_RUNS = True
RUN_LOCAL = True       # Runs models sequentially and waits; potentially slow.
SUBMIT_SLURM = False    # Submits all prepared models and returns job IDs.

if RUN_LOCAL and SUBMIT_SLURM:
    raise ValueError('Choose local execution or Slurm submission, not both.')


configs = build_configs(
    MASSES_MSUN,
    initial_y=INITIAL_Y,
    initial_z=INITIAL_Z,
    central_h1_limit=CENTRAL_H1_LIMIT,
    stage=STAGE,
    rgb_radius_limit=RGB_RADIUS_LIMIT,
    partition=PARTITION,
    walltime=WALLTIME,
    cpus_per_task=CPUS_PER_TASK,
)

for config in configs:
    print(f'{config.mass_msun:8g} M_sun  ->  {model_label(config.stage, config.mass_msun, config.initial_z)}')

template_text = (TEMPLATE_DIR / 'inlist_project').read_text()
preview = render_inlist_project(template_text, configs[0])
keys_to_show = (
    'filename_for_profile_when_terminate', 'save_pulse_data_filename',
    'initial_mass', 'initial_y', 'initial_z', 'Zbase',
    'xa_central_lower_limit(1)', 'rgb_radius_limit',
)
for line in preview.splitlines():
    if any(key in line for key in keys_to_show):
        print(line)

assert f"'{model_label(configs[0].stage, configs[0].mass_msun, configs[0].initial_z)}.profile'" in preview
assert f"'{model_label(configs[0].stage, configs[0].mass_msun, configs[0].initial_z)}.mesa'" in preview
print('In-memory smoke test passed.')

runs = []
if PREPARE_RUNS:
    runs = prepare_grid(
        configs, TEMPLATE_DIR, OUTPUT_ROOT, overwrite=OVERWRITE
    )
    print(f'Prepared {len(runs)} run(s) under {OUTPUT_ROOT}')
else:
    print('PREPARE_RUNS is false; no directories were written.')

inspect_runs(runs)

job_ids = {}
if RUN_LOCAL:
    for run in runs:
        print(f'Running {run.label} ...', flush=True)
        run_local(run)
elif SUBMIT_SLURM:
    for run in runs:
        job_ids[run.label] = submit_slurm(run)
    print(json.dumps(job_ids, indent=2))
else:
    print('Setup only: no MESA processes or Slurm jobs were started.')

status = inspect_runs(runs)
print(json.dumps(status, indent=2))

if runs:
    metadata = json.loads(runs[0].metadata_path.read_text())
    print('\nFirst run metadata:')
    print(json.dumps(metadata, indent=2))

    if runs[0].gyre_path.is_file():
        fields = runs[0].gyre_path.read_text().splitlines()[0].split()
        names = ['zones', 'mass_g', 'radius_cm', 'luminosity_erg_s', 'version']
        print('\nGYRE header:', dict(zip(names, fields)))

MESA_DIR = /Users/mchau/mesa
Project root: /Users/mchau/mesa_tutorial
       8 M_sun  ->  RGB8Msun_Z0p0142
   filename_for_profile_when_terminate = 'RGB8Msun_Z0p0142.profile'
   save_pulse_data_filename = 'RGB8Msun_Z0p0142.mesa'
   initial_mass = 8
   initial_y = 0.2703  ! Helium mass fraction
   initial_z = 0.0142  ! Metal mass fraction, hydrogen = 1 - metal - helium
   xa_central_lower_limit(1) = 0.0
  use_Zbase_for_Type1 = .true.
  Zbase = 0.0142
In-memory smoke test passed.
Prepared 1 run(s) under /Users/mchau/mesa_tutorial/results/mams_profiles
Running RGB8Msun_Z0p0142 ...
[
  {
    "label": "RGB8Msun_Z0p0142",
    "prepared": false,
    "mesa_profile": false,
    "gyre_model": false,
    "run_dir": "/Users/mchau/mesa_tutorial/results/mams_profiles/RGB8Msun_Z0p0142"
  }
]


FileNotFoundError: [Errno 2] No such file or directory: '/Users/mchau/mesa_tutorial/results/mams_profiles/RGB8Msun_Z0p0142/metadata.json'